# 20 — W3 ProRank reranker on the dev set

Per RecSys_Challenge_Plan §A5 + §4 (W3 gate). Adds the ProRank last-token-logit-diff reranker on top of the W2 CMQR pipeline. Tries three configurations and reports nDCG@10 vs the exp-021 champion.

**Pipeline:**
1. **Mount Drive** + persistent caches.
2. **Clone** fresh-model branch, install deps + vLLM.
3. **Pytest gate** — confirm pro_rank/state_tracker/cmqr/reward_fns/build_trl_datasets all green.
4. **Smoke** ProRank (1 min on A100).
5. **Run inference** with the W3 config (`110-prorank-rerank-devset`).
6. **Score** via `evaluate_devset.py` and check the gate.
7. **Optional:** if base ProRank under-performs, run the BGE cut-path eval as a fallback baseline.
8. **Stretch:** GRPO training cell (commented; run separately if W3 gate fails).

**W3 gate:** dev nDCG@10 ≥ 0.0934 (champion 0.0784 + 0.015).
**Base ProRank caveat:** paper §3.2 reports base Qwen-0.5B BEIR ≈ 0.30, trained ≈ 0.51. **Inference-only may not pass the gate** — keep the BGE cut-path cell ready.
**Wall time on A100:** ~30–45 min for the W3 config (CMQR + ProRank scoring of 100 candidates × 8000 turns).

In [ ]:
# 3) HF auth — required for push_to_hub.
#
# Setup: Colab → 🔑 Secrets pane → add `HF_TOKEN` with WRITE scope.
# Get the token at https://huggingface.co/settings/tokens.
#
# Fail-fast: aborts immediately if the secret is missing or the token is
# invalid — better than failing 3 hours into training when push_to_hub fires.
import os, sys
from google.colab import userdata
from huggingface_hub import whoami

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        f"\u274c HF_TOKEN secret not found in Colab ({e!r}).\n"
        f"   1) Open the \U0001f511 Secrets pane in the left sidebar.\n"
        f"   2) Add a secret named exactly `HF_TOKEN` (case-sensitive).\n"
        f"   3) Toggle 'Notebook access' ON for this notebook."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

try:
    user = whoami(token=HF_TOKEN)
    HF_USERNAME = user["name"]
    os.environ["HF_USERNAME"] = HF_USERNAME
    print(f"\u2713 HF auth ok \u2014 logged in as {HF_USERNAME}")
except Exception as e:
    raise SystemExit(
        f"\u274c HF auth failed: {e!r}\n"
        f"   Token may lack WRITE scope. Regenerate at https://huggingface.co/settings/tokens"
    )

In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'

In [ ]:
# 2b) Mount Drive + persistent caches (same convention as 02 / 10).
import os, shutil
from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}\nretrying with force_remount=True ...')
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/prorank_dev']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

print(f'datasets cache  : {os.environ["HF_DATASETS_CACHE"]}')
print(f'experiments dir : {EXPECTED_CACHE} -> {os.readlink(EXPECTED_CACHE)}')

In [ ]:
# 3) Install deps. ProRank loads via transformers; CMQR uses vLLM.
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf bm25s sentence-transformers peft
!pip install -q --upgrade vllm
!python -c 'import torch, transformers, vllm, peft; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "vllm", vllm.__version__, "peft", peft.__version__)'

In [ ]:
# 4) Pytest pre-flight — confirm all module-level tests pass before burning A100 time.
!cd /content/recsys2026 && python -m pytest tests/test_reward_fns.py tests/test_state_tracker.py tests/test_cmqr.py tests/test_pro_rank.py tests/test_build_trl_datasets.py -q

In [ ]:
# 5) ProRank smoke — confirm Qwen-0.5B loads and scores correctly.
!cd /content/recsys2026 && python scripts/smoke_pro_rank.py --device cuda

In [ ]:
# 6) W3 inference — CMQR + ProRank on full devset.
TID = '110-prorank-rerank-devset'
BATCH_SIZE = 16
DEVICE = 'cuda'
ATTN = 'sdpa'

!nvidia-smi --query-gpu=memory.total,memory.used,memory.free --format=csv
!cd /content/recsys2026/music-crs-baselines && python run_inference_devset.py \
    --tid {TID} --batch_size {BATCH_SIZE} --device {DEVICE} --attn_implementation {ATTN}

In [ ]:
# 7) Score with the official evaluator.
!cd /content/recsys2026/music-crs-evaluator && python evaluate_devset.py --tid {TID}

In [ ]:
# 8) Read scores + check the W3 gate.
import json
from pathlib import Path

scores_path = Path(f'/content/recsys2026/music-crs-evaluator/exp/scores/devset/{TID}.json')
with scores_path.open() as f:
    scores = json.load(f)

ndcg10 = scores.get('ndcg@10', 0.0)
ndcg20 = scores.get('ndcg@20', 0.0)
ndcg1 = scores.get('ndcg@1', 0.0)
catdiv = scores.get('catalog_diversity', 0.0)
lexdiv = scores.get('lexical_diversity', 0.0)

CHAMPION_NDCG10 = 0.0784
GATE_THRESHOLD = CHAMPION_NDCG10 + 0.015
W2_FALLBACK = 0.0834   # nDCG@10 a successful W2 produces; ProRank ≥ this is incremental win

print('=' * 60)
print('W3 ProRank — DEV SET RESULTS')
print('=' * 60)
print(f'  nDCG@1  : {ndcg1:.4f}')
print(f'  nDCG@10 : {ndcg10:.4f}  (champion {CHAMPION_NDCG10}, gate ≥ {GATE_THRESHOLD})')
print(f'  nDCG@20 : {ndcg20:.4f}')
print(f'  CatDiv  : {catdiv:.4f}')
print(f'  LexDiv  : {lexdiv:.4f}')
print()
delta = ndcg10 - CHAMPION_NDCG10
print(f'  Δ vs champion nDCG@10  : {delta:+.4f}')
print(f'  Δ vs W2 baseline       : {ndcg10 - W2_FALLBACK:+.4f}')
print()
print('GATE (plan §4 W3 row):')
# P1 #8 fix: explicit flag controls whether Cell 11 (BGE cut-path) runs.
BGE_CUT_PATH_NEEDED = ndcg10 < GATE_THRESHOLD
GRPO_TRAINING_NEEDED = ndcg10 < W2_FALLBACK
if ndcg10 >= GATE_THRESHOLD:
    print(f'  PASS   {ndcg10:.4f} ≥ {GATE_THRESHOLD} — proceed to W4 (KTO warmup)')
    print('  → Cell 11 (BGE cut-path) will SKIP (gate already passed).')
elif ndcg10 >= W2_FALLBACK:
    print(f'  WEAK PASS  {ndcg10:.4f} ≥ {W2_FALLBACK} (W2 baseline) but < gate;')
    print('  → Cell 11 (BGE cut-path) will RUN and pick the higher.')
else:
    print(f'  FAIL   {ndcg10:.4f} < {W2_FALLBACK} — base ProRank regressed retrieval')
    print('  Likely diagnosis: base Qwen-0.5B uncalibrated for music (paper §3.2).')
    print('  → Cell 11 (BGE cut-path) will RUN.')
    print('  → Cell 13 (GRPO training) recommended in parallel.')
print('=' * 60)
print(f'\n  BGE_CUT_PATH_NEEDED   = {BGE_CUT_PATH_NEEDED}')
print(f'  GRPO_TRAINING_NEEDED  = {GRPO_TRAINING_NEEDED}')

In [ ]:
# 9) Save artifacts to Drive.
import shutil
pred_src = f'/content/recsys2026/music-crs-baselines/exp/inference/devset/{TID}.json'
scores_src = f'/content/recsys2026/music-crs-evaluator/exp/scores/devset/{TID}.json'
drive_dst = f'{DRIVE_BASE}/prorank_dev'
shutil.copy(pred_src, f'{drive_dst}/{TID}.json')
shutil.copy(scores_src, f'{drive_dst}/scores_{TID}.json')
print(f'predictions: {drive_dst}/{TID}.json')
print(f'scores:      {drive_dst}/scores_{TID}.json')
!ls -lh {drive_dst}/

In [ ]:
# 10) ProRank diagnostics — sample the score cache to see what the model is doing.
from pathlib import Path
import pickle
from collections import Counter

cache_files = list(Path(f'{DRIVE_BASE}/experiments_cache/prorank').glob('*.pkl'))
if not cache_files:
    cache_files = list(Path('/content/recsys2026/music-crs-baselines/experiments/cache/prorank').glob('*.pkl'))
for fp in cache_files:
    with fp.open('rb') as f:
        cache = pickle.load(f)
    scores = list(cache.values())
    print(f'{fp.name}: {len(cache):,} cached (q,d) scores')
    if scores:
        import statistics
        print(f'  score mean: {statistics.mean(scores):.3f}  std: {statistics.stdev(scores):.3f}')
        print(f'  range: [{min(scores):.3f}, {max(scores):.3f}]')
        print('  positive scores ("yes" wins):  '
              f'{sum(1 for s in scores if s > 0):,} ({100*sum(1 for s in scores if s > 0)/len(scores):.1f}%)')
        print('  negative scores ("no" wins):   '
              f'{sum(1 for s in scores if s < 0):,} ({100*sum(1 for s in scores if s < 0)/len(scores):.1f}%)')

## Cell 11 — BGE cut-path (run only if Cell 8 says FAIL or WEAK PASS)

If the base-Qwen ProRank under-performs, switch to BGE-reranker-v2-m3 (already in the repo, battle-tested 568M cross-encoder, no training required). This is the documented cut-path from plan §A5.

Build a sibling config that swaps `reranker_type: pro_rank` → `bge_reranker_v2_m3`, run the same inference + eval, and compare.

In [ ]:
# 11) BGE cut-path — runs ONLY if the gate failed. (P1 #8 fix)
if not BGE_CUT_PATH_NEEDED:
    print('Skipping BGE cut-path: ProRank gate already passed.')
else:
    BGE_TID = '111-bge-rerank-cmqr-devset'
    yaml_src = '/content/recsys2026/music-crs-baselines/config/110-prorank-rerank-devset.yaml'
    yaml_dst = f'/content/recsys2026/music-crs-baselines/config/{BGE_TID}.yaml'
    with open(yaml_src) as f:
        yml = f.read()
    yml = yml.replace('reranker_type: "pro_rank"', 'reranker_type: "bge_reranker_v2_m3"')
    with open(yaml_dst, 'w') as f:
        f.write(yml)
    print(f'wrote {yaml_dst}')

    !cd /content/recsys2026/music-crs-baselines && python run_inference_devset.py \
        --tid {BGE_TID} --batch_size {BATCH_SIZE} --device {DEVICE} --attn_implementation {ATTN}
    !cd /content/recsys2026/music-crs-evaluator && python evaluate_devset.py --tid {BGE_TID}

In [ ]:
# 12) Compare ProRank vs BGE side-by-side (only if BGE was actually run).
import json
from pathlib import Path

if BGE_CUT_PATH_NEEDED:
    for tid in [TID, BGE_TID]:
        p = Path(f'/content/recsys2026/music-crs-evaluator/exp/scores/devset/{tid}.json')
        if p.exists():
            with p.open() as f:
                s = json.load(f)
            print(f'\n{tid}:')
            print(f'  nDCG@10 = {s.get("ndcg@10", 0):.4f}')
            print(f'  nDCG@20 = {s.get("ndcg@20", 0):.4f}')
            print(f'  CatDiv  = {s.get("catalog_diversity", 0):.4f}')
else:
    print(f'BGE cut-path skipped — ProRank passed gate at nDCG@10 = {ndcg10:.4f}')

## Cell 13 — GRPO training stretch (only if both ProRank-base and BGE underwhelm)

If neither inference-only path passes the W3 gate, train ProRank via GRPO using the verifiable nDCG@10 reward from `scripts/reward_fns.r_retr`. This is the paper-faithful path (~2 A100-hr).

Approach (skill `huggingface-llm-trainer`):
- Base model: Qwen-2.5-0.5B-Instruct
- LoRA r=32, alpha=32 (plan §6.3 standard)
- Reward: `r_retr(reranked_top20, gold_track_id)` from `scripts/reward_fns.py`
- Group size G=2, ~4k optimizer steps
- Trackio: `report_to="trackio"`, group="a-stage"
- Push adapter to Hub: `hub_model_id="orrimoch/recsys2026-prorank-{date}"`

Cell 13 below is a placeholder — uncomment + flesh out only after the inference-only path is shown to be insufficient.

In [ ]:
# 13) GRPO training stretch — DEFERRED. Honest spec replaces the prior 0.0
# placeholder (P1 #7 fix). DO NOT execute as-is — full ProRank GRPO is more
# than the TRL GRPOTrainer one-liner; it needs a custom training loop because
# our policy is a *scorer* (logit-diff), not a generator.
#
# Required pieces (paper §3.2):
#   1. Per training step: sample a query from the prompt set.
#   2. For each of the K candidates from CMQR's top-100, run the policy to
#      get score = logit('yes') - logit('no').
#   3. Sort candidates by score → reranked top-20.
#   4. Compute nDCG@10 against the gold track ID for the query.
#      Use `scripts/reward_fns.r_retr` (the leaderboard nDCG fn) — it's
#      already battle-tested on the W1 anchor parquet.
#   5. The "reward" for this query is the resulting nDCG@10. Each candidate
#      contributes a per-token gradient via the relative-policy ratio.
#   6. Group-relative advantage (G=2): repeat the rollout twice and use the
#      score difference as the credit signal.
#
# Why TRL GRPOTrainer doesn't drop in:
#   - GRPOTrainer expects (prompt, completion) text generation. Our policy
#     produces a SCALAR per (prompt, candidate) pair, not a completion.
#   - The reward is a function of the FULL ranked list (nDCG@10), not a
#     per-completion scalar.
#
# Cleanest implementation paths:
#   (a) Custom training loop using PEFT + manual policy/reference KL.
#       ~150 LOC, follows DeepSpeed-Chat or trl's GRPOTrainer source as
#       a template. This is the paper-faithful path.
#   (b) Repurpose TRL RewardTrainer with PAIRWISE comparisons (positive vs
#       hard-negative track for the same query). Simpler, off-the-shelf,
#       loses the nDCG-as-reward formulation. Recommended starting point
#       if (a) blows up the time budget.
#
# Reference: documents/research/ProRank_2506.03487.pdf §3.2.
# Skill reference: .claude/skills/huggingface-llm-trainer/scripts/train_grpo_example.py
#
# When you're ready to train, EITHER:
#   - Implement (a) in a separate notebook (e.g. colab/21_train_prorank_grpo.ipynb)
#   - Or implement (b) inline below (uncomment + adapt scripts/train_dpo_example.py
#     pattern from the installed skill).

print("Cell 13: GRPO training stretch — see comments above for the plan.")
print("Run only if both ProRank (Cell 7) AND BGE (Cell 11) miss the W3 gate.")
print("Recommended starting point: TRL RewardTrainer with pairwise positives/negatives.")

## After the run

**ProRank PASS (`nDCG@10 ≥ 0.0934`):**
- Update `documents/RecSys_Challenge_Plan.md` with the W3 result.
- Append a row to `documents/submissions_log.md` (`[dev-local]` tag).
- Proceed to **W4 — KTO warmup** (B1 stage of Component B).

**ProRank WEAK PASS (`0.0834 ≤ nDCG@10 < 0.0934`):**
- Run Cell 11 (BGE cut-path) and pick the higher one.
- If still below gate, ship the higher of the two to W4 + run Cell 13 (GRPO training) in parallel on a side Colab session.

**ProRank FAIL (`nDCG@10 < 0.0834`):**
- Confirm with Cell 11 (BGE cut-path).
- If BGE also fails, run Cell 13 (GRPO training) — paper §3.2 expects ~+0.2 nDCG lift over the base SLM.
- Document failure in `documents/experiments_log.md`.